# Анализ пользовательского поведения и ключевых метрик маркетплейса

Интернет-маркетплейс собирает данные о заказах, оплатах и активности пользователей.
Компания сталкивается с проблемами:
- снижение конверсии в покупку
- низкий уровень возврата пользователей
- неравномерное распределение выручки

Для повышения эффективности бизнеса необходимо проанализировать пользовательское поведение, выявить узкие места в воронке и предложить продуктовые улучшения.

----

<b>Цель проекта:</b>

Провести комплексный анализ маркетплейса и выявить ключевые факторы, влияющие на конверсию, удержание пользователей и выручку.
Сформировать продуктовые гипотезы для роста ключевых метрик.

---

<b>Задачи проекта</b>

- Изучить и подготовить данные (EDA)
- Проанализировать воронку заказов (conversion funnel)
- Оценить удержание пользователей (retention / повторные покупки)
- Рассчитать ключевые метрики:
    - Conversion Rate
    - Retention
    - ARPU / средний чек
- Выполнить сегментацию пользователей (новые / возвращающиеся / высокоценные)
- Построить аналитический дашборд
- Сформулировать продуктовые выводы и рекомендации

---

<b>Описание данных</b>

Датасет: Brazilian E-Commerce Public Dataset by Olist

- olist_customers_dataset — пользователи                                                                                        
`customer_id` — уникальный идентификатор пользователя                        
`customer_unique_id` — уникальный пользователь (может делать несколько заказов)                       
`customer_city` — город                 
`customer_state` — регион                           

- olist_orders_dataset — заказы                           
`order_id` — идентификатор заказа                        
`customer_id` — идентификатор пользователя                             
`order_status` — статус заказа (created / approved / delivered / canceled и др.)                             
`order_purchase_timestamp` — время создания заказа                         
`order_approved_at` — время подтверждения оплаты                       
`order_delivered_carrier_date` — дата передачи заказа в доставку                      
`order_delivered_customer_date` — дата доставки заказа клиенту                                
`order_estimated_delivery_date` — ожидаемая дата доставки                    

- olist_order_items_dataset — товары в заказе                      
`order_id` — идентификатор заказа                               
`order_item_id` — номер товара внутри заказа                  
`product_id` — идентификатор товара                  
`seller_id` — идентификатор продавца                  
`shipping_limit_date` — крайний срок передачи товара в доставку                  
`price` — стоимость товара                  
`freight_value` — стоимость доставки                                 

- olist_order_payments_dataset — платежи                                   
`order_id` — идентификатор заказа                  
`payment_sequential` — номер платежа внутри заказа                  
`payment_type` — способ оплаты                  
`payment_installments` — количество платежей / рассрочка                  
`payment_value` — сумма платежа (BRL)                                     

Все денежные значения представлены в бразильских реалах (BRL).

<img src="HRhd2Y0.png" width="800">

<a id="11"></a>
## Загрузка данных

In [2]:
import pandas as pd

In [3]:
customers = pd.read_csv("olist_customers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")

In [5]:
def show_df_info(df, name='DataFrame'):
    print(f'\n=== {name} ===')

    print('\nИнформация о данных:')
    df.info() 

    print('\nПервые строки:')
    display(df.head())

    print('\nОписательная статистика:')
    display(df.describe(include='all'))

In [6]:
show_df_info(customers, 'Данные о пользователях')


=== Данные о пользователях ===

Информация о данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB

Первые строки:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



Описательная статистика:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
count,99441,99441,99441.000000,99441,99441
unique,99441,96096,NaN,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,NaN,sao paulo,SP
freq,1,17,NaN,15540,41746
mean,NaN,NaN,35137.474583,NaN,NaN
std,NaN,NaN,29797.938996,NaN,NaN
min,NaN,NaN,1003.000000,NaN,NaN
25%,NaN,NaN,11347.000000,NaN,NaN
50%,NaN,NaN,24416.000000,NaN,NaN
75%,NaN,NaN,58900.000000,NaN,NaN


Таблица содержит 99441 строку без пропусков. 96 096 уникальных пользователей, больше всего пользователей из Сан-Пауло.

In [9]:
show_df_info(orders, 'Данные о заказах')


=== Данные о заказах ===

Информация о данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB

Первые строки:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Описательная статистика:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-04-11 10:48:14,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 23:38:46,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


Таблица содержит 99441 строку. Есть пропуски в полях `order_approved_at`(время подтверждения оплаты), `order_delivered_carrier_date` (дата передачи заказа в доставку) и `order_delivered_customer_date` (дата доставки). Временные столбцы необходимо перевести в datetime формат. 

In [8]:
show_df_info(items, 'Данные о товарах в заказах')


=== Данные о товарах в заказах ===

Информация о данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB

Первые строки:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14



Описательная статистика:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
count,112650,112650.000000,112650,112650,112650,112650.000000,112650.000000
unique,98666,NaN,32951,3095,93318,NaN,NaN
top,8272b63d03f5f79c56e9e4120aec44ef,NaN,aca2eb7d00ea1a7b8ebd4e68314663af,6560211a19b47992c3666cc44a7e94c0,2017-07-21 18:25:23,NaN,NaN
freq,21,NaN,527,2033,21,NaN,NaN
mean,NaN,1.197834,NaN,NaN,NaN,120.653739,19.990320
std,NaN,0.705124,NaN,NaN,NaN,183.633928,15.806405
min,NaN,1.000000,NaN,NaN,NaN,0.850000,0.000000
25%,NaN,1.000000,NaN,NaN,NaN,39.900000,13.080000
50%,NaN,1.000000,NaN,NaN,NaN,74.990000,16.260000
75%,NaN,1.000000,NaN,NaN,NaN,134.900000,21.150000


Таблица содержит 112650 строк без пропусков. Временные столбцы необходимо перевести в datetime формат. Средняя стоимость товара - 120 бразильских реала. Средняя стоимость доставки - 20 реалов. Есть товары с бесплатной доставкой. Самый дешевый товар стоит 0.85 реала, самый дорогой 6 735 реалов.

In [12]:
show_df_info(payments, 'Данные о платежах')


=== Данные о платежах ===

Информация о данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB

Первые строки:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45



Описательная статистика:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
count,103886,103886.000000,103886,103886.000000,103886.000000
unique,99440,NaN,5,NaN,NaN
top,fa65dad1b0e818e3ccc5cb0e39231352,NaN,credit_card,NaN,NaN
freq,29,NaN,76795,NaN,NaN
mean,NaN,1.092679,NaN,2.853349,154.100380
std,NaN,0.706584,NaN,2.687051,217.494064
min,NaN,1.000000,NaN,0.000000,0.000000
25%,NaN,1.000000,NaN,1.000000,56.790000
50%,NaN,1.000000,NaN,1.000000,100.000000
75%,NaN,1.000000,NaN,4.000000,171.837500


Таблица содержит 103886 строк без пропусков. Средняя стоимость товара - 120 бразильских реала. Средняя стоимость доставки - 20 реалов. Есть товары с бесплатной доставкой. Самый дешевый товар стоит 0.85 реала, самый дорогой 6 735 реалов.